# Q01 · Xu hướng mức độ nghiêm trọng

**Phụ trách:** A · **Tiêu chí:** D2, D3, D4

Tỷ lệ vụ nghiêm trọng thay đổi thế nào theo tháng và năm khi giữ cố định cơ cấu loại đường?

## 1. Động cơ và phạm vi

**Người dùng kết quả:** người lập kế hoạch an toàn giao thông muốn biết tỷ lệ vụ nghiêm trọng (có người chết hoặc bị thương nặng) đang tăng hay giảm, và mức thay đổi đó có thật hay chỉ do tai nạn dịch chuyển giữa các loại đường.

**Vì sao phải giữ cố định cơ cấu loại đường:** tỷ lệ nghiêm trọng khác nhau rõ giữa các loại đường (vòng xuyến thấp, đường đơn cao). Nếu tỷ trọng vụ trên đường đơn tăng thì tỷ lệ chung sẽ tăng dù trên từng loại đường không có gì thay đổi. So chuỗi tỷ lệ thô với chuỗi đã chuẩn hóa theo loại đường cho phép tách hai nguồn thay đổi này.

**Đơn vị quan sát:** một vụ tai nạn được cảnh sát ghi nhận (bảng `accidents_clean.csv`).

**Phạm vi và giới hạn:**
- Từ 01/2005 đến 12/2015, toàn Vương quốc Anh (dữ liệu STATS19).
- Kết quả mô tả **các vụ được ghi nhận**, không phải rủi ro trên mỗi chuyến đi: không có dữ liệu lưu lượng giao thông.
- Đây là phân tích mô tả, không chứng minh nguyên nhân.

**Tiêu chí độ sâu:**
- D2: gom theo tháng/năm, pivot, trung bình trượt 12 tháng.
- D3: tỷ lệ chuẩn hóa theo loại đường.
- D4: kiểm soát cơ cấu loại đường.
- D5: khoảng tin cậy bootstrap.
- D6: kiểm tra độ vững với các lựa chọn trọng số, xử lý mã "không rõ" và khung thời gian.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Notebook có thể chạy từ thư mục gốc hoặc từ thư mục notebooks/.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if not (project_root / "environment.yml").is_file():
    raise FileNotFoundError("Mở Jupyter Notebook từ thư mục gốc của dự án.")

raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"

# Màu và kiểu biểu đồ dùng chung
BLUE, BLUE_LIGHT, ORANGE = "#2a78d6", "#86b6ef", "#eb6834"
INK, INK_2, GRID = "#0b0b0b", "#52514e", "#e6e5e0"
plt.rcParams.update({
    "figure.figsize": (9, 4.5), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": INK_2, "axes.labelcolor": INK_2,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlecolor": INK,
    "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "lines.linewidth": 2, "legend.frameon": False,
})

In [ ]:
data_path = processed_dir / "accidents_clean.csv"
required_columns = ['accident_id', 'accident_date', 'accident_year', 'road_type', 'is_severe']

# Đọc vài dòng để kiểm tra bảng bàn giao trước khi xử lý dữ liệu thật.
if data_path.is_file():
    preview = pd.read_csv(data_path, nrows=5, dtype={"accident_id": "string"})
    missing_columns = sorted(set(required_columns) - set(preview.columns))
    if missing_columns:
        raise ValueError(f"{data_path.name} thiếu cột {missing_columns}")
    print(f"Đã kiểm tra các cột cần dùng trong {data_path.name}.")
else:
    print(f"Chưa có {data_path.name}. Chờ bảng do notebook 01 tạo.")

## 2. Chọn mẫu và kiểm tra dữ liệu

Các bước:
1. Đọc 5 cột cần dùng, kiểm tra `accident_id` duy nhất và không có giá trị thiếu.
2. Kiểm tra khoảng thời gian, đảm bảo đủ 132 tháng.
3. Xử lý `road_type = 9` ("Unknown" theo bảng mã `Road_Type.csv`). Mã này không phải -1 nên bước làm sạch chung chưa đổi thành NA. **Phân tích chính loại các vụ này**, vì không gán được trọng số loại đường có nghĩa. Mục 4.4 kiểm tra lại khi giữ chúng thành một nhóm riêng.
4. Không lọc theo `is_severe`: mẫu số là toàn bộ vụ trong kỳ.

In [ ]:
accident_df = pd.read_csv(
    data_path,
    usecols=required_columns,
    dtype={"accident_id": "string"},
    parse_dates=["accident_date"],
)
assert accident_df["accident_id"].is_unique, "accident_id bị trùng"
print("Số dòng đọc vào:", f"{len(accident_df):,}")
print("Giá trị thiếu theo cột:")
print(accident_df.isna().sum().to_string())

accident_df["month"] = accident_df["accident_date"].dt.to_period("M")
print("\nKhoảng thời gian:", accident_df["accident_date"].min().date(), "→",
      accident_df["accident_date"].max().date())
print("Số tháng có dữ liệu:", accident_df["month"].nunique())
assert (accident_df["accident_date"].dt.year == accident_df["accident_year"]).all()

In [ ]:
ROAD_LABELS = (
    pd.read_csv(raw_dir / "contextCSVs" / "Road_Type.csv")
    .set_index("code")["label"]
    .to_dict()
)
road_summary = (
    accident_df.groupby("road_type")["is_severe"]
    .agg(so_vu="size", so_vu_nghiem_trong="sum", ty_le="mean")
    .assign(ty_trong=lambda t: t["so_vu"] / t["so_vu"].sum())
)
road_summary.index = road_summary.index.map(lambda c: f"{c} · {ROAD_LABELS.get(c, '?')}")
road_summary.style.format({"so_vu": "{:,}", "so_vu_nghiem_trong": "{:,}",
                           "ty_le": "{:.2%}", "ty_trong": "{:.2%}"})

In [ ]:
n_before = len(accident_df)
df = accident_df[accident_df["road_type"] != 9].copy()
print(f"Loại road_type = 9 (Unknown): {n_before:,} → {len(df):,} dòng "
      f"(bỏ {n_before - len(df):,}, {(n_before - len(df)) / n_before:.2%})")

# Ô nhỏ nhất khi chia tháng × loại đường: kiểm tra trước khi tính tỷ lệ
cell_size = df.groupby(["month", "road_type"]).size().unstack()
print("Ô tháng × loại đường bị trống:", int(cell_size.isna().sum().sum()))
print("Số vụ nhỏ nhất trong một ô tháng × loại đường:", int(cell_size.min().min()))

## 3. Phương pháp

**Tỷ lệ thô** của kỳ $t$ (tháng hoặc năm):

$$\text{thô}_t = \frac{\text{số vụ nghiêm trọng trong kỳ } t}{\text{số vụ trong kỳ } t} = \sum_k s_{k,t}\, p_{k,t}$$

- $p_{k,t}$: tỷ lệ nghiêm trọng của loại đường $k$ trong kỳ $t$.
- $s_{k,t}$: tỷ trọng vụ trên loại đường $k$ trong kỳ $t$.

Tỷ lệ thô thay đổi khi $p$ thay đổi **hoặc** khi $s$ thay đổi.

**Tỷ lệ chuẩn hóa trực tiếp (D3, D4)** thay $s_{k,t}$ bằng một bộ trọng số cố định $w_k$:

$$\text{chuẩn hóa}_t = \sum_k w_k\, p_{k,t}$$

- Phân tích chính dùng $w_k$ là cơ cấu loại đường của cả giai đoạn 2005–2015.
- Khi đó chuỗi chuẩn hóa chỉ thay đổi do $p_{k,t}$, tức do mức nghiêm trọng trong từng loại đường.
- Chênh lệch giữa chuỗi thô và chuỗi chuẩn hóa là phần do **cơ cấu** gây ra.

**Thời gian (D2):** pivot bảng kỳ × loại đường, gom theo năm và theo tháng. Chuỗi tháng có mùa vụ nên thêm trung bình trượt 12 tháng để thấy xu hướng.

**Độ bất định (D5):** bootstrap phân tầng.
- Rút lại có hoàn lại các vụ trong từng ô (kỳ × loại đường), lặp B = 2000 lần, rồi tính lại tỷ lệ chuẩn hóa.
- `is_severe` là biến 0/1, nên số vụ nghiêm trọng của một ô rút lại có phân phối Binomial($n_{k,t}$, $\hat p_{k,t}$). Vì vậy chỉ cần sinh số Binomial cho từng ô thay vì rút lại 1,77 triệu dòng.
- Khoảng tin cậy 95% lấy theo phân vị 2,5% – 97,5%.

**Độ vững (D6):** tính lại chênh lệch 2015 – 2005 khi:
- dùng trọng số năm 2005 hoặc năm 2015;
- giữ `road_type = 9` thành nhóm riêng;
- so trung bình 3 năm đầu với 3 năm cuối thay vì chỉ hai năm đầu–cuối.

In [ ]:
B = 2000
rng = np.random.default_rng(42)


def cell_table(data, period):
    """Pivot số vụ (n) và số vụ nghiêm trọng (k) theo kỳ × loại đường."""
    g = data.groupby([period, "road_type"])["is_severe"].agg(n="size", k="sum")
    return g["n"].unstack(fill_value=0), g["k"].unstack(fill_value=0)


def rates(data, period, weights=None):
    """Trả về tỷ lệ thô, tỷ lệ chuẩn hóa và bảng p_{k,t}."""
    n, k = cell_table(data, period)
    if weights is None:
        weights = data["road_type"].value_counts(normalize=True)
    w = weights.reindex(n.columns)
    assert np.isclose(w.sum(), 1) and (n > 0).all().all()
    p = k / n
    crude = k.sum(axis=1) / n.sum(axis=1)
    std = (p * w).sum(axis=1)
    return crude, std, p


def bootstrap_std(data, period, weights=None, B=B, rng=rng):
    """Bootstrap phân tầng trong từng ô kỳ × loại đường. Trả về mảng (B, số kỳ)."""
    n, k = cell_table(data, period)
    if weights is None:
        weights = data["road_type"].value_counts(normalize=True)
    w = weights.reindex(n.columns).values
    p_boot = rng.binomial(n.values, (k / n).values, size=(B, *n.shape)) / n.values
    return pd.DataFrame((p_boot * w).sum(axis=2), columns=n.index)


def ci(samples, axis=0):
    return np.percentile(samples, [2.5, 97.5], axis=axis)

In [ ]:
# Kiểm tra trung gian: tỷ lệ thô tính bằng công thức Σ s·p phải trùng với trung bình is_severe
crude_y, std_y, p_y = rates(df, "accident_year")
assert np.allclose(crude_y, df.groupby("accident_year")["is_severe"].mean())

weights_all = df["road_type"].value_counts(normalize=True).sort_index()
print("Trọng số cố định w_k (cơ cấu 2005–2015):")
print(weights_all.rename(index=ROAD_LABELS).map("{:.2%}".format).to_string())

# Cơ cấu loại đường theo năm (pivot, D2): có dịch chuyển nhiều không?
share_y = pd.crosstab(df["accident_year"], df["road_type"], normalize="index")
share_y.rename(columns=ROAD_LABELS).style.format("{:.2%}")

## 4. Kết quả

### 4.1. Theo năm: tỷ lệ thô và tỷ lệ chuẩn hóa

In [ ]:
boot_y = bootstrap_std(df, "accident_year")
lo_y, hi_y = ci(boot_y)
year_table = pd.DataFrame({
    "so_vu": df.groupby("accident_year").size(),
    "ty_le_tho": crude_y,
    "ty_le_chuan_hoa": std_y,
    "CI95_thap": lo_y,
    "CI95_cao": hi_y,
})
year_table["phan_do_co_cau"] = year_table["ty_le_tho"] - year_table["ty_le_chuan_hoa"]
year_table.style.format({"so_vu": "{:,}", "ty_le_tho": "{:.2%}", "ty_le_chuan_hoa": "{:.2%}",
                         "CI95_thap": "{:.2%}", "CI95_cao": "{:.2%}",
                         "phan_do_co_cau": "{:+.3%}"})

In [ ]:
fig, ax = plt.subplots()
years = year_table.index
ax.fill_between(years, lo_y * 100, hi_y * 100, color=BLUE, alpha=0.15, linewidth=0,
                label="CI 95% (bootstrap) của tỷ lệ chuẩn hóa")
ax.plot(years, std_y * 100, color=BLUE, marker="o", markersize=5,
        label="Chuẩn hóa theo loại đường")
ax.plot(years, crude_y * 100, color=ORANGE, linestyle="--", marker="o", markersize=3,
        label="Tỷ lệ thô")
for y_val, txt, dy in [(std_y.iloc[-1], "Chuẩn hóa", -0.12), (crude_y.iloc[-1], "Thô", 0.12)]:
    ax.annotate(f"{txt} {y_val:.1%}", (years[-1], y_val * 100), xytext=(8, dy * 40),
                textcoords="offset points", va="center", fontsize=9, color=INK_2)
ax.set_title("Tỷ lệ vụ nghiêm trọng theo năm, 2005–2015")
ax.set_xlabel("Năm")
ax.set_ylabel("Vụ nghiêm trọng / tổng số vụ (%)")
ax.set_xticks(years)
ax.set_xlim(years[0] - 0.3, years[-1] + 1.3)
ax.legend(loc="upper left", fontsize=9)
ax.text(0, -0.2, f"n = {len(df):,} vụ (đã loại road_type = 9). "
        "Trọng số chuẩn hóa: cơ cấu loại đường 2005–2015.",
        transform=ax.transAxes, fontsize=8, color=INK_2)
plt.tight_layout()
plt.show()

In [ ]:
diff_boot = boot_y[2015] - boot_y[2005]
diff_lo, diff_hi = ci(diff_boot)
crude_diff = crude_y[2015] - crude_y[2005]
std_diff = std_y[2015] - std_y[2005]
print(f"Chênh lệch 2015 − 2005, tỷ lệ thô:       {crude_diff * 100:+.2f} điểm %")
print(f"Chênh lệch 2015 − 2005, tỷ lệ chuẩn hóa: {std_diff * 100:+.2f} điểm %  "
      f"(CI 95%: {diff_lo * 100:+.2f} đến {diff_hi * 100:+.2f})")
print(f"Phần do thay đổi cơ cấu loại đường:      {(crude_diff - std_diff) * 100:+.3f} điểm %")
print(f"Thay đổi tương đối (chuẩn hóa):          {std_diff / std_y[2005]:+.1%}")

### 4.2. Theo tháng: chuỗi chuẩn hóa và trung bình trượt 12 tháng

Trung bình trượt 12 tháng loại bỏ mùa vụ trong năm. Mỗi điểm là trung bình của 12 tháng gần nhất, nên 11 tháng đầu không có giá trị.

In [ ]:
crude_m, std_m, p_m = rates(df, "month", weights=weights_all)
boot_m = bootstrap_std(df, "month", weights=weights_all)
lo_m, hi_m = ci(boot_m)
month_ts = pd.DataFrame({"tho": crude_m, "chuan_hoa": std_m, "lo": lo_m, "hi": hi_m})
month_ts.index = month_ts.index.to_timestamp()
month_ts["truot_12"] = month_ts["chuan_hoa"].rolling(12).mean()

print("Độ rộng CI 95% trung bình: theo tháng "
      f"{(month_ts['hi'] - month_ts['lo']).mean() * 100:.2f} điểm %, theo năm "
      f"{(hi_y - lo_y).mean() * 100:.2f} điểm %")
print("Chênh lệch lớn nhất giữa thô và chuẩn hóa theo tháng: "
      f"{(month_ts['tho'] - month_ts['chuan_hoa']).abs().max() * 100:.3f} điểm %")

# Mùa vụ: tỷ lệ chuẩn hóa trung bình theo tháng trong năm
season = month_ts.groupby(month_ts.index.month)["chuan_hoa"].mean()
print("Tháng có tỷ lệ chuẩn hóa trung bình cao nhất:", season.idxmax(),
      f"({season.max():.2%}); thấp nhất:", season.idxmin(), f"({season.min():.2%})")

In [ ]:
fig, ax = plt.subplots()
ax.fill_between(month_ts.index, month_ts["lo"] * 100, month_ts["hi"] * 100,
                color=BLUE, alpha=0.12, linewidth=0, label="CI 95% (bootstrap) theo tháng")
ax.plot(month_ts.index, month_ts["chuan_hoa"] * 100, color=BLUE_LIGHT, linewidth=1.2,
        label="Tỷ lệ chuẩn hóa theo tháng")
ax.plot(month_ts.index, month_ts["truot_12"] * 100, color=BLUE, linewidth=2.2,
        label="Trung bình trượt 12 tháng")
ax.set_title("Tỷ lệ vụ nghiêm trọng chuẩn hóa theo loại đường, theo tháng")
ax.set_xlabel("Tháng")
ax.set_ylabel("Vụ nghiêm trọng / tổng số vụ (%)")
ax.legend(loc="upper left", fontsize=9)
ax.text(0, -0.2, f"132 tháng, mỗi tháng {df.groupby('month').size().min():,}–"
        f"{df.groupby('month').size().max():,} vụ. Trọng số: cơ cấu loại đường 2005–2015.",
        transform=ax.transAxes, fontsize=8, color=INK_2)
plt.tight_layout()
plt.show()

### 4.3. Mức thay đổi trong từng loại đường (D4)

Nếu xu hướng là thật chứ không do cơ cấu thì nó phải xuất hiện **bên trong** từng loại đường. Mỗi loại đường được bootstrap độc lập cho năm 2005 và năm 2015.

In [ ]:
n_y, k_y = cell_table(df, "accident_year")
rows = []
for rt in n_y.columns:
    n05, n15 = n_y.loc[2005, rt], n_y.loc[2015, rt]
    p05, p15 = k_y.loc[2005, rt] / n05, k_y.loc[2015, rt] / n15
    d_boot = rng.binomial(n15, p15, B) / n15 - rng.binomial(n05, p05, B) / n05
    lo, hi = ci(d_boot)
    rows.append({"road_type": rt, "loai_duong": ROAD_LABELS[rt], "n_2005": n05, "n_2015": n15,
                 "p_2005": p05, "p_2015": p15, "chenh_lech": p15 - p05, "CI95_thap": lo, "CI95_cao": hi})
by_road = pd.DataFrame(rows).set_index("road_type")
by_road.style.format({"n_2005": "{:,}", "n_2015": "{:,}", "p_2005": "{:.2%}", "p_2015": "{:.2%}",
                      "chenh_lech": "{:+.2%}", "CI95_thap": "{:+.2%}", "CI95_cao": "{:+.2%}"})

In [ ]:
plot_df = by_road.sort_values("chenh_lech")
fig, ax = plt.subplots(figsize=(9, 4))
ypos = np.arange(len(plot_df))
ax.errorbar(plot_df["chenh_lech"] * 100, ypos,
            xerr=[(plot_df["chenh_lech"] - plot_df["CI95_thap"]) * 100,
                  (plot_df["CI95_cao"] - plot_df["chenh_lech"]) * 100],
            fmt="o", color=BLUE, markersize=8, elinewidth=2, capsize=0)
ax.axvline(std_diff * 100, color=INK_2, linestyle=":", linewidth=1.2)
ax.annotate(f"Toàn bộ (chuẩn hóa) {std_diff * 100:+.2f}", (std_diff * 100, len(plot_df) - 0.5),
            xytext=(4, 0), textcoords="offset points", fontsize=8, color=INK_2)
ax.axvline(0, color=INK_2, linewidth=1)
ax.set_yticks(ypos)
ax.set_yticklabels([f"{r.loai_duong}\n(n 2015 = {r.n_2015:,})" for r in plot_df.itertuples()],
                   fontsize=9)
ax.grid(axis="y", visible=False)
ax.set_title("Thay đổi tỷ lệ vụ nghiêm trọng 2005 → 2015 theo loại đường")
ax.set_xlabel("Chênh lệch tỷ lệ 2015 − 2005 (điểm %) · thanh ngang: CI 95% bootstrap")
plt.tight_layout()
plt.show()

### 4.4. Kiểm tra độ vững (D6)

Tính lại chênh lệch của tỷ lệ chuẩn hóa dưới các lựa chọn hợp lý khác. Nếu dấu và độ lớn gần như không đổi thì kết luận không phụ thuộc vào lựa chọn của phân tích chính.

In [ ]:
def diff_with_ci(data, weights=None, early=(2005,), late=(2015,)):
    _, std, _ = rates(data, "accident_year", weights)
    boot = bootstrap_std(data, "accident_year", weights)
    est = std[list(late)].mean() - std[list(early)].mean()
    d = boot[list(late)].mean(axis=1) - boot[list(early)].mean(axis=1)
    lo, hi = ci(d)
    return est, lo, hi


w_2005 = df.loc[df["accident_year"] == 2005, "road_type"].value_counts(normalize=True)
w_2015 = df.loc[df["accident_year"] == 2015, "road_type"].value_counts(normalize=True)
variants = {
    "Chính: loại mã 9, trọng số 2005–2015, so 2015 với 2005": diff_with_ci(df),
    "Trọng số cơ cấu năm 2005": diff_with_ci(df, w_2005),
    "Trọng số cơ cấu năm 2015": diff_with_ci(df, w_2015),
    "Giữ mã 9 (Unknown) thành nhóm riêng": diff_with_ci(accident_df),
    "So trung bình 2013–2015 với 2005–2007": diff_with_ci(df, early=(2005, 2006, 2007),
                                                          late=(2013, 2014, 2015)),
}
robust = pd.DataFrame(variants, index=["chenh_lech", "CI95_thap", "CI95_cao"]).T
robust["CI_chua_0"] = (robust["CI95_thap"] <= 0) & (robust["CI95_cao"] >= 0)
robust.style.format({"chenh_lech": "{:+.2%}", "CI95_thap": "{:+.2%}", "CI95_cao": "{:+.2%}"})

## 5. Trả lời và giới hạn

> **Phần này để bạn tự viết.** Theo mục 6 của đề bài và cam kết trong `LLM_USAGE.md`, phần trả lời và kết luận phải do người phụ trách tự viết. Dưới đây là các ý cần có. Lấy số cụ thể từ output của mục 4.

**Trả lời câu hỏi:**
- Theo năm, tỷ lệ thô và tỷ lệ chuẩn hóa thay đổi thế nào từ 2005 đến 2015 (mục 4.1)? Có kèm khoảng tin cậy không?
- Phần chênh lệch do cơ cấu loại đường lớn hay nhỏ? Điều đó nói gì về câu hỏi "giữ cố định cơ cấu"?
- Xu hướng có xuất hiện trong từng loại đường không (mục 4.3)? Loại đường nào thay đổi nhiều nhất, khoảng tin cậy có chứa 0 không?
- Theo tháng: trung bình trượt cho thấy gì? Dao động theo tháng có vượt độ rộng CI không (mục 4.2)?
- Kết luận có đổi dưới các lựa chọn ở mục 4.4 không?

**Ý nghĩa thực tế:** người lập kế hoạch nên đọc kết quả này thế nào? Điều gì bất ngờ?

**Giới hạn cần nêu:**
- Dữ liệu chỉ gồm vụ được ghi nhận. Tai nạn nhẹ dễ bị bỏ sót hơn tai nạn nghiêm trọng, nên nếu tỷ lệ bỏ sót vụ nhẹ thay đổi theo thời gian thì tỷ lệ nghiêm trọng thay đổi theo mà không phản ánh rủi ro thật.
- Cách cảnh sát phân loại "bị thương nặng" có thể thay đổi theo thời gian. Bootstrap không bắt được loại sai lệch hệ thống này.
- Bootstrap giả định các vụ độc lập. Các vụ cùng khu vực hoặc cùng tháng có thể tương quan, nên CI có thể hơi hẹp hơn thực tế.
- Đã loại 0,6% vụ có `road_type = 9`. Mục 4.4 cho thấy việc này ảnh hưởng thế nào.
- Chỉ giữ cố định loại đường. Các yếu tố khác như giới hạn tốc độ hay khu vực thành thị/nông thôn có thể cũng đã dịch chuyển.